# Experimento 2. Autenticación básica con Keycloak

> **Objetivo:** Entender cómo funciona Identity & Access Management (IAM) usando Keycloak + FastAPI.

| Concepto | Descripción |
|---|---|
| **Keycloak** | Servidor de identidad open-source (IAM) |
| **OAuth2** | Protocolo de autorización |
| **OpenID Connect** | Capa de identidad sobre OAuth2 |
| **JWT** | JSON Web Token — credencial firmada digitalmente |
| **FastAPI** | Backend en Python que protege endpoints con JWT |

---

##Tabla de contenidos

1. [Arquitectura del sistema](#arquitectura)
2. [Levantar Keycloak con Docker](#docker)
3. [Configurar el Realm en Keycloak](#realm)
4. [Crear usuarios y roles](#usuarios)
5. [Backend FastAPI protegido](#fastapi)
6. [Probar el flujo completo](#pruebas)
7. [Qué aprendimos](#conclusiones)

## 1.Arquitectura del sistema

```
┌─────────────┐     (1) Login (usuario/contraseña)
│   Cliente   │ ─────────────────────────────────────► ┌─────────────┐
│  (curl /    │                                         │  Keycloak   │
│   Postman)  │ ◄─────────────────────────────────────  │  :8080      │
└─────────────┘     (2) JWT (access_token)              └─────────────┘
       │                                                       │
       │ (3) GET /private                                      │ JWKS (claves públicas)
       │     Authorization: Bearer <JWT>                       │
       ▼                                                       │
┌─────────────┐     (4) Validar JWT con JWKS ◄─────────────────┘
│  FastAPI    │
│  :8000      │ ──► (5) 200 OK  {"mensaje": "Bienvenido"}
└─────────────┘      o  401 Unauthorized
```

**Flujo resumido:**
1. El usuario hace login en Keycloak → recibe un **JWT**
2. Envía el JWT en el header `Authorization: Bearer ...` al backend
3. FastAPI verifica la firma del JWT con las claves públicas de Keycloak (JWKS)
4. Si es válido → acceso permitido; si no → 401

## 2. Levantar Keycloak con Docker

### 2.1 `docker-compose.yml`

Creacion de archivo:

```yaml
version: '3.8'

services:
  keycloak:
    image: quay.io/keycloak/keycloak:24.0.1
    container_name: keycloak
    command: start-dev
    environment:
      KC_BOOTSTRAP_ADMIN_USERNAME: admin
      KC_BOOTSTRAP_ADMIN_PASSWORD: admin
    ports:
      - "8080:8080"
    networks:
      - auth-net

  fastapi:
    build: ./api
    container_name: fastapi_app
    ports:
      - "8000:8000"
    depends_on:
      - keycloak
    networks:
      - auth-net

networks:
  auth-net:
    driver: bridge
```

### 2.2 Iniciar los servicios

In [ ]:
#Bloque de referencia para documentar el comando

comando = "docker compose up -d"
print(f"▶ Ejecutar en terminal: {comando}")
print()
print("Keycloak tarda ~30 segundos en arrancar.")
print("Accede al panel en: http://localhost:8080")
print("Usuario: admin | Contraseña: admin")

In [ ]:
import time
import urllib.request

KEYCLOAK_URL = "http://localhost:8080"

def esperar_keycloak(max_intentos=15, espera=5):
    """Espera a que Keycloak esté listo."""
    print("⏳ Esperando a que Keycloak arranque...")
    for intento in range(1, max_intentos + 1):
        try:
            req = urllib.request.urlopen(f"{KEYCLOAK_URL}/health/ready", timeout=3)
            if req.status == 200:
                print(f"Keycloak listo en el intento {intento}")
                return True
        except Exception:
            print(f"  Intento {intento}/{max_intentos} — no disponible aún...")
            time.sleep(espera)
    print("Keycloak no respondió. Verifica que Docker esté corriendo.")
    return False

esperar_keycloak()

## 3.Configurar el Realm en Keycloak

### 3.1 Obtener token de administrador

In [ ]:
import requests

KEYCLOAK_URL  = "http://localhost:8080"
ADMIN_USER    = "admin"
ADMIN_PASS    = "admin"
REALM_NAME    = "monarca"
CLIENT_ID_API = "api"

def obtener_token_admin():
    """Obtiene el access token del realm master para usar la Admin API."""
    url = f"{KEYCLOAK_URL}/realms/master/protocol/openid-connect/token"
    resp = requests.post(url, data={
        "client_id":  "admin-cli",
        "username":   ADMIN_USER,
        "password":   ADMIN_PASS,
        "grant_type": "password",
    })
    resp.raise_for_status()
    token = resp.json()["access_token"]
    print("Token de administrador obtenido")
    return token

admin_token = obtener_token_admin()
headers_admin = {"Authorization": f"Bearer {admin_token}", "Content-Type": "application/json"}

### 3.2 Crear el Realm `monarca`

In [ ]:
import json

def crear_realm(realm_name: str):
    """Crea un nuevo realm en Keycloak."""
    url  = f"{KEYCLOAK_URL}/admin/realms"
    body = {
        "realm":   realm_name,
        "enabled": True,
        "displayName": realm_name.capitalize(),
    }
    resp = requests.post(url, headers=headers_admin, json=body)
    if resp.status_code == 201:
        print(f"Realm '{realm_name}' creado exitosamente")
    elif resp.status_code == 409:
        print(f"Realm '{realm_name}' ya existe — continuando...")
    else:
        resp.raise_for_status()

crear_realm(REALM_NAME)

### 3.3 Crear el Client `api`

In [ ]:
def crear_client(realm_name: str, client_id: str):
    """Crea un client público para la API en el realm."""
    url  = f"{KEYCLOAK_URL}/admin/realms/{realm_name}/clients"
    body = {
        "clientId":                  client_id,
        "enabled":                   True,
        "publicClient":              True,           # Sin client_secret
        "directAccessGrantsEnabled": True,           # Permite password flow
        "standardFlowEnabled":       True,
        "redirectUris":              ["*"],
        "webOrigins":                ["*"],
    }
    resp = requests.post(url, headers=headers_admin, json=body)
    if resp.status_code == 201:
        print(f"Client '{client_id}' creado")
    elif resp.status_code == 409:
        print(f"Client '{client_id}' ya existe — continuando...")
    else:
        resp.raise_for_status()

crear_client(REALM_NAME, CLIENT_ID_API)

## 4.Crear usuarios y roles

Vamos a crear dos roles (`admin`, `user`) y dos usuarios de prueba.

In [ ]:
ROLES = ["admin", "user"]

def crear_rol(realm_name: str, nombre_rol: str):
    """Crea un rol en el realm."""
    url  = f"{KEYCLOAK_URL}/admin/realms/{realm_name}/roles"
    resp = requests.post(url, headers=headers_admin, json={"name": nombre_rol})
    if resp.status_code == 201:
        print(f"Rol '{nombre_rol}' creado")
    elif resp.status_code == 409:
        print(f"Rol '{nombre_rol}' ya existe")
    else:
        resp.raise_for_status()

print("Creando roles...")
for rol in ROLES:
    crear_rol(REALM_NAME, rol)

In [ ]:
USUARIOS = [
    {"username": "ana",   "password": "ana123",   "rol": "admin"},
    {"username": "pablo", "password": "pablo123", "rol": "user"},
]

def crear_usuario(realm_name: str, username: str, password: str) -> str:
    """Crea un usuario y devuelve su ID."""
    url  = f"{KEYCLOAK_URL}/admin/realms/{realm_name}/users"
    body = {
        "username":  username,
        "enabled":   True,
        "credentials": [{"type": "password", "value": password, "temporary": False}],
    }
    resp = requests.post(url, headers=headers_admin, json=body)
    if resp.status_code == 201:
        # El ID está en el header Location
        user_id = resp.headers["Location"].split("/")[-1]
        print(f"Usuario '{username}' creado (id: {user_id})")
        return user_id
    elif resp.status_code == 409:
        # Ya existe — obtener su ID
        search = requests.get(
            f"{url}?username={username}", headers=headers_admin
        ).json()
        user_id = search[0]["id"]
        print(f"Usuario '{username}' ya existe (id: {user_id})")
        return user_id
    else:
        resp.raise_for_status()

def asignar_rol(realm_name: str, user_id: str, nombre_rol: str):
    """Asigna un rol de realm a un usuario."""
    # 1. Obtener representación del rol
    rol_resp = requests.get(
        f"{KEYCLOAK_URL}/admin/realms/{realm_name}/roles/{nombre_rol}",
        headers=headers_admin
    ).json()
    # 2. Asignarlo al usuario
    url = f"{KEYCLOAK_URL}/admin/realms/{realm_name}/users/{user_id}/role-mappings/realm"
    requests.post(url, headers=headers_admin, json=[rol_resp]).raise_for_status()
    print(f"     → Rol '{nombre_rol}' asignado")

print("Creando usuarios y asignando roles...")
for u in USUARIOS:
    uid = crear_usuario(REALM_NAME, u["username"], u["password"])
    asignar_rol(REALM_NAME, uid, u["rol"])

## 5.Backend FastAPI protegido

Crea la carpeta `api/` con los siguientes archivos:

### 5.1 Estructura de archivos

```
proyecto/
├── docker-compose.yml
└── api/
    ├── Dockerfile
    ├── requirements.txt
    └── main.py
```

### 5.2 `api/requirements.txt`

```
fastapi==0.111.0
uvicorn==0.29.0
python-jose[cryptography]==3.3.0
PyJWT[crypto]==2.8.0
httpx==0.27.0
requests==2.31.0
```

### 5.3 `api/Dockerfile`

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY main.py .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

In [ ]:
# api/main.py — Contenido completo
# Copia este código en api/main.py

codigo_main = '''
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2AuthorizationCodeBearer
from jwt import PyJWKClient
import jwt
import os

KEYCLOAK_URL = os.getenv("KEYCLOAK_URL", "http://keycloak:8080")
REALM        = os.getenv("REALM", "monarca")
CLIENT_ID    = os.getenv("CLIENT_ID", "api")

BASE = f"{KEYCLOAK_URL}/realms/{REALM}/protocol/openid-connect"

app = FastAPI(title="API Protegida con Keycloak")

oauth2_scheme = OAuth2AuthorizationCodeBearer(
    tokenUrl=f"{BASE}/token",
    authorizationUrl=f"{BASE}/auth",
)

def verificar_token(token: str = Depends(oauth2_scheme)) -> dict:
    """Valida el JWT con las claves públicas de Keycloak (JWKS)."""
    jwks_client = PyJWKClient(f"{BASE}/certs")
    try:
        signing_key = jwks_client.get_signing_key_from_jwt(token)
        datos = jwt.decode(
            token,
            signing_key.key,
            algorithms=["RS256"],
            audience=CLIENT_ID,
            options={"verify_aud": False},  # Keycloak public clients
        )
        return datos
    except jwt.exceptions.ExpiredSignatureError:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED,
                            detail="Token expirado")
    except jwt.exceptions.InvalidTokenError as e:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED,
                            detail=f"Token inválido: {e}")

@app.get("/")
def raiz():
    return {"mensaje": "API pública OK"}

@app.get("/public")
def publico():
    """Endpoint abierto — no requiere autenticación."""
    return {"mensaje": "Este endpoint es público"}

@app.get("/private")
def privado(usuario: dict = Depends(verificar_token)):
    """Endpoint protegido — requiere JWT válido."""
    roles = usuario.get("realm_access", {}).get("roles", [])
    return {
        "mensaje":  f"Bienvenido, {usuario.get(\'preferred_username\')}!",
        "roles":    roles,
        "sub":      usuario.get("sub"),
    }

@app.get("/admin-only")
def solo_admin(usuario: dict = Depends(verificar_token)):
    """Endpoint exclusivo para administradores."""
    roles = usuario.get("realm_access", {}).get("roles", [])
    if "admin" not in roles:
        raise HTTPException(status_code=status.HTTP_403_FORBIDDEN,
                            detail="Se requiere rol admin")
    return {"mensaje": "Área de administración", "usuario": usuario.get("preferred_username")}
'''

print("=" * 60)
print("api/main.py")
print("=" * 60)
print(codigo_main)

## 6. Flujo completo

### Paso previo: levantar todo
```bash
docker compose up --build -d
```

In [ ]:
# ── Prueba 1: Obtener JWT haciendo login como usuario normal (pablo) ──

def login(username: str, password: str, realm: str = REALM_NAME) -> dict:
    """Simula el login de un usuario y retorna los tokens."""
    url  = f"{KEYCLOAK_URL}/realms/{realm}/protocol/openid-connect/token"
    resp = requests.post(url, data={
        "client_id":  CLIENT_ID_API,
        "username":   username,
        "password":   password,
        "grant_type": "password",
    })
    resp.raise_for_status()
    return resp.json()

print("Login como 'pablo' (rol: user)...")
tokens_pablo = login("pablo", "pablo123")
print(f" access_token (primeros 80 chars): {tokens_pablo['access_token'][:80]}...")
print(f"  Token type: {tokens_pablo['token_type']}")
print(f"  Expira en:  {tokens_pablo['expires_in']} segundos")

In [ ]:
import base64, json as jsonlib

def decodificar_jwt_sin_verificar(token: str) -> dict:
    """Decodifica el payload del JWT (solo para inspección, sin verificar firma)."""
    partes  = token.split(".")
    payload = partes[1]
    # Añadir padding si falta
    payload += "=" * (4 - len(payload) % 4)
    return jsonlib.loads(base64.urlsafe_b64decode(payload))

print("🔍 Payload del JWT de pablo:")
payload = decodificar_jwt_sin_verificar(tokens_pablo["access_token"])
campos_importantes = ["preferred_username", "realm_access", "exp", "iss", "sub"]
for campo in campos_importantes:
    print(f"  {campo}: {payload.get(campo)}")

In [ ]:
# ── Prueba 2: Endpoint público (sin token) ──

API_URL = "http://localhost:8000"

resp = requests.get(f"{API_URL}/public")
print(f"GET /public → {resp.status_code}")
print(f"  Respuesta: {resp.json()}")

In [ ]:
# ── Prueba 3: Endpoint privado CON token válido (pablo) ──

headers_pablo = {"Authorization": f"Bearer {tokens_pablo['access_token']}"}
resp = requests.get(f"{API_URL}/private", headers=headers_pablo)
print(f"GET /private (pablo, token válido) → {resp.status_code}")
print(f"  Respuesta: {resp.json()}")

In [ ]:
# ── Prueba 4: Endpoint privado SIN token → debe dar 401 ──

resp = requests.get(f"{API_URL}/private")
print(f"GET /private (sin token) → {resp.status_code}")
print(f"  Respuesta: {resp.json()}")
assert resp.status_code == 401, "Esperábamos 401 Unauthorized"
print("Correcto: acceso denegado sin token")

In [ ]:
# ── Prueba 5: Endpoint /admin-only ──

# Pablo (rol: user) → debe dar 403
resp = requests.get(f"{API_URL}/admin-only", headers=headers_pablo)
print(f"GET /admin-only (pablo, rol=user) → {resp.status_code}")
print(f"  Respuesta: {resp.json()}")

print()

# Ana (rol: admin) → debe dar 200
tokens_ana = login("ana", "ana123")
headers_ana = {"Authorization": f"Bearer {tokens_ana['access_token']}"}
resp = requests.get(f"{API_URL}/admin-only", headers=headers_ana)
print(f"GET /admin-only (ana, rol=admin)  → {resp.status_code}")
print(f"  Respuesta: {resp.json()}")

In [ ]:
# ── Prueba 6: Token falso → debe rechazarlo ──

resp = requests.get(f"{API_URL}/private",
                    headers={"Authorization": "Bearer token.falso.aqui"})
print(f"GET /private (token falso) → {resp.status_code}")
print(f"  Respuesta: {resp.json()}")
assert resp.status_code == 401
print("Correcto: token falso rechazado")

### 6.1 Resumen de pruebas esperadas

| Prueba | Endpoint | Token | Código esperado | Resultado |
|---|---|---|---|---|
| 1 | `/public` | ninguno | `200` | ✅ |
| 2 | `/private` | válido (pablo) | `200` | ✅ |
| 3 | `/private` | ninguno | `401` | ✅ |
| 4 | `/private` | falso | `401` | ✅ |
| 5 | `/admin-only` | pablo (user) | `403` | ✅ |
| 6 | `/admin-only` | ana (admin) | `200` | ✅ |

##Referencias

- [Keycloak Documentation](https://www.keycloak.org/documentation)
- [FastAPI + Keycloak — imjoseangel](https://github.com/imjoseangel/fastapi-keycloak)
- [Keycloak Node.js example — v-ladynev](https://github.com/v-ladynev/keycloak-nodejs-example)
- [RFC 7519 — JSON Web Token](https://datatracker.ietf.org/doc/html/rfc7519)
- [OpenID Connect spec](https://openid.net/connect/)